# 05 - Test-time Unicode defenses

This notebook evaluates preprocessing as a deployment-time wrapper. Models remain trained on the original clean text; only evaluation inputs are normalized or repaired, which separates defense effects from retraining effects.

**Inputs:** Clean and attacked conditions, trained model artifacts, and undefended predictions.  
**Outputs:** Defended prediction partitions and an illustrative repair table.


## Setup and defense definitions

The notebook locally defines the five configured defenses, prediction helpers for both model families, and the shared artifact layout. NFC and NFKC perform standard normalization; stripping removes default-ignorable code points; the full repair additionally reverses the curated English confusable map.

In [1]:
import os
import unicodedata
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import regex
import yaml

# Same path setup as notebook 00; repeated for independent execution.
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
CONFIG = yaml.safe_load((ROOT / 'config/experiment.yaml').read_text(encoding='utf-8'))

PATHS = {key: ROOT / value for key, value in CONFIG['paths'].items()}
for directory in (PATHS['predictions'], PATHS['tables'],):
    directory.mkdir(parents=True, exist_ok=True)

# Same curated mapping as notebook 02; repeated for independent execution.
HOMOGLYPH_MAP = {
    'A': ('Α', 'А'), 'B': ('Β', 'В'), 'C': ('Ϲ', 'С'), 'E': ('Ε', 'Е'),
    'H': ('Η', 'Н'), 'I': ('Ι', 'І'), 'J': ('Ј',), 'K': ('Κ', 'К'),
    'M': ('Μ', 'М'), 'N': ('Ν',), 'O': ('Ο', 'О'), 'P': ('Ρ', 'Р'),
    'S': ('Ѕ',), 'T': ('Τ', 'Т'), 'X': ('Χ', 'Х'), 'Y': ('Υ', 'У'), 'Z': ('Ζ',),
    'a': ('а',), 'c': ('с',), 'e': ('е',), 'h': ('һ',), 'i': ('і',),
    'j': ('ј',), 'k': ('κ',), 'o': ('ο', 'о'), 'p': ('р',), 's': ('ѕ',),
    'x': ('х',), 'y': ('у',),
}

CONFUSABLE_REVERSE = {candidate: source for source, candidates in HOMOGLYPH_MAP.items() for candidate in candidates}

DEFAULT_IGNORABLE = regex.compile(r'\p{Default_Ignorable_Code_Point}+')

def remove_default_ignorables(text):
    return DEFAULT_IGNORABLE.sub('', text)

def repair_confusables_english(text):
    return ''.join(CONFUSABLE_REVERSE.get(ch, ch) for ch in text)

def apply_defense(text, defense):
    if defense == 'none':
        return text
    if defense == 'nfc':
        return unicodedata.normalize('NFC', text)
    if defense == 'nfkc':
        return unicodedata.normalize('NFKC', text)
    if defense == 'strip_ignorables':
        return remove_default_ignorables(text)
    if defense == 'nfkc_strip':
        return remove_default_ignorables(unicodedata.normalize('NFKC', text))
    if defense == 'nfkc_strip_confusable':
        normalized = remove_default_ignorables(unicodedata.normalize('NFKC', text))
        return repair_confusables_english(normalized)
    raise ValueError(f'Unknown defense: {defense}')

# Same BPE tokenizer and prediction helper as notebook 03; repeated for independent execution.
import tiktoken

_BPE_ENCODING = tiktoken.get_encoding('o200k_base')

def bpe_tokenize(text):
    return [f'tok_{token_id}' for token_id in _BPE_ENCODING.encode(text)]

def predict_sklearn(model, texts, labels, chunk_size=None):
    chunk_size = chunk_size or CONFIG['classical']['prediction_chunk_size']
    predictions = []
    probabilities = []
    labels_array = np.asarray(labels, dtype=int)
    classes = np.asarray(model.classes_, dtype=int)
    class_to_column = {label: idx for idx, label in enumerate(classes)}
    for start in range(0, len(texts), chunk_size):
        stop = min(start + chunk_size, len(texts))
        batch = list(texts[start:stop])
        batch_proba = model.predict_proba(batch)
        batch_pred = classes[np.argmax(batch_proba, axis=1)]
        gold_columns = np.array([class_to_column[int(label)] for label in labels_array[start:stop]])
        predictions.append(batch_pred)
        probabilities.append(batch_proba[np.arange(stop - start), gold_columns])
    return np.concatenate(predictions), np.concatenate(probabilities)

PREDICTION_COLUMNS = [
    'dataset', 'example_id', 'model', 'model_seed', 'attack', 'strength', 'attack_seed',
    'defense', 'gold', 'prediction', 'gold_probability', 'correct', 'changed_count',
]

# Same prediction layout as notebook 03.
def prediction_partition_path(source, dataset, model, model_seed, defense):
    return PATHS['predictions'] / source / dataset / model / f'seed_{model_seed}' / f'{defense}.parquet'

def write_prediction_partition(frame, source, dataset, model, model_seed, defense):
    path = prediction_partition_path(source, dataset, model, model_seed, defense)
    path.parent.mkdir(parents=True, exist_ok=True)
    frame[PREDICTION_COLUMNS].to_parquet(path, index=False)
    return path


## Build defended conditions

The same clean-plus-attacked condition matrix used for raw evaluation is rebuilt for each dataset. A defense transforms only the text column, leaving example identity, attack metadata, labels, and change counts untouched.

In [2]:
os.environ.setdefault('PYTORCH_ENABLE_MPS_FALLBACK', '1')
import gc
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

data = pd.read_parquet(PATHS['processed_data'])
attacks = pd.read_parquet(PATHS['attacks'])
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')

# Same evaluation conditions as notebook 03.
def condition_frame(dataset_name):
    clean = data[(data['dataset'] == dataset_name) & (data['split'] == 'test')][['dataset', 'example_id', 'text', 'label']].copy()
    clean = clean.assign(attack='clean', strength=0.0, attack_seed=-1, changed_count=0)
    attacked = attacks[attacks['dataset'] == dataset_name][['dataset', 'example_id', 'attacked_text', 'label', 'attack', 'strength', 'attack_seed', 'changed_count']].rename(columns={'attacked_text': 'text'})
    return pd.concat([clean, attacked], ignore_index=True)


## Reuse unchanged predictions

Raw predictions are aligned by example and condition and reused for unchanged text. Each model evaluates only distinct transformed text-label pairs, then those predictions are expanded back to the original row order.


In [3]:
def predict_transformer(model, tokenizer, texts, labels):
    predictions, probabilities = [], []
    batch_size = CONFIG['transformer']['eval_batch_size']
    model.to(device)
    model.eval()
    with torch.inference_mode():
        for start in range(0, len(texts), batch_size):
            batch = texts[start:start + batch_size]
            encoded = tokenizer(batch, padding=True, truncation=True,
                                max_length=CONFIG['transformer']['max_length'], return_tensors='pt')
            encoded = {key: value.to(device) for key, value in encoded.items()}
            logits = model(**encoded).logits.float().cpu()
            probability = torch.softmax(logits, dim=1).numpy()
            predictions.append(probability.argmax(axis=1))
            probabilities.append(probability[np.arange(len(batch)), labels[start:start + batch_size]])
    return np.concatenate(predictions), np.concatenate(probabilities)


## Evaluate missing defense partitions

Existing partitions are reused. For missing partitions, both model families follow the same alignment, deduplication, and writing steps; only model loading and inference differ.


In [4]:
models = [(name, CONFIG['project']['root_seed']) for name in ['word_tfidf', 'char_tfidf', 'bpe_tfidf']]
models += [('distilbert', seed) for seed in CONFIG['transformer']['seeds']]
keys = ['dataset', 'example_id', 'attack', 'strength', 'attack_seed']
created = reused = 0

for dataset_name in ['banking77', 'trec']:
    base = condition_frame(dataset_name)
    for defense in CONFIG['defenses']['names']:
        conditions = base.assign(text=base['text'].map(lambda text: apply_defense(text, defense)))
        changed = base['text'].ne(conditions['text']).to_numpy()
        for model_name, model_seed in models:
            target = prediction_partition_path('defense', dataset_name, model_name, model_seed, defense)
            if target.exists():
                reused += 1
                continue

            raw = pd.read_parquet(prediction_partition_path('raw', dataset_name, model_name, model_seed, 'none'))
            aligned = base[keys + ['label']].merge(
                raw[keys + ['gold', 'prediction', 'gold_probability']],
                on=keys, how='left', validate='one_to_one', sort=False,
            )
            assert aligned['gold'].notna().all() and aligned['label'].eq(aligned['gold']).all(), 'Raw predictions do not align'
            pred = aligned['prediction'].to_numpy(copy=True)
            probability = aligned['gold_probability'].to_numpy(copy=True)
            if changed.any():
                unique = conditions.loc[changed, ['text', 'label']].drop_duplicates()
                texts, labels = unique['text'].tolist(), unique['label'].to_numpy()
                if model_name == 'distilbert':
                    model_path = PATHS['models'] / 'transformer' / dataset_name / f'seed_{model_seed}' / 'final'
                    tokenizer = AutoTokenizer.from_pretrained(model_path)
                    model = AutoModelForSequenceClassification.from_pretrained(model_path)
                    changed_pred, changed_probability = predict_transformer(model, tokenizer, texts, labels)
                else:
                    model_path = PATHS['models'] / 'classical' / dataset_name / f'{model_name}.joblib'
                    model = joblib.load(model_path)
                    changed_pred, changed_probability = predict_sklearn(model, texts, labels)
                unique = unique.assign(prediction=changed_pred, gold_probability=changed_probability)
                expanded = conditions.loc[changed, ['text', 'label']].merge(
                    unique, on=['text', 'label'], how='left', validate='many_to_one', sort=False,
                )
                pred[changed] = expanded['prediction']
                probability[changed] = expanded['gold_probability']
                del model
                gc.collect()
                if torch.backends.mps.is_available():
                    torch.mps.empty_cache()

            frame = conditions.drop(columns='text').rename(columns={'label': 'gold'}).assign(
                model=model_name, model_seed=model_seed, defense=defense,
                prediction=pred, gold_probability=probability,
            )
            frame['correct'] = frame['gold'].eq(frame['prediction'])
            write_prediction_partition(frame, 'defense', dataset_name, model_name, model_seed, defense)
            created += 1

print(f'Defense partitions: {created} created, {reused} reused.')


Defense partitions: 0 created, 60 reused.


In [5]:
examples = attacks[(attacks['strength'] == CONFIG['analysis']['primary_strength']) & (attacks['changed_count'] > 0)]
examples = examples.groupby('attack', sort=False).head(1)[['attack', 'clean_text', 'attacked_text']].copy()
examples['repaired_text'] = examples['attacked_text'].map(lambda text: apply_defense(text, 'nfkc_strip_confusable'))
examples


,attack,clean_text,attacked_text,repaired_text
6,homoglyph,How do I locate my card?,Hοw do I locate my сard?,How do I locate my card?
18,invisible,How do I locate my card?,Ho‍w do I​ l﻿ocate my card​?,How do I locate my card?
30,compatibility,How do I locate my card?,How dｏ I ｌocate ｍy carｄ?,How do I locate my card?
54,mixed,How do I locate my card?,How ｄо Ｉ locate my​ card?,How do I locate my card?


## Interpretation

Defense recovery must be considered alongside the change in clean performance. NFKC and ignorable stripping follow Unicode properties, whereas confusable repair encodes English-specific assumptions and can alter legitimate text in other languages or domains.